# Ingest — Azure AI Foundry

Two sources, because cost and usage live in different systems:

| | |
|---|---|
| **Cost Management Query API** | spend and usage quantity, daily, by meter |
| **Azure Monitor Metrics** | token counts and provisioned-capacity utilisation |

Both tables are **optional**. Load neither and the Foundry page is simply empty, which is a
supported state. Spend supplies recorded cost and meter quantities; Monitor adds token/request
counts and PTU utilisation. The shipped cost-per-million measure assumes all selected billing
quantities are in millions of tokens: it is not valid for mixed units or provisioned hours.

**Source conventions to check in your tenant** (original observations from 2026-08-07):

- The shipped Foundry cost measure filters specifically to **`Foundry Models`**. Legacy/other
  service names can be collected but are not included in that card. No currency conversion is performed.
- The Cost Management dimension is **`Meter`**, not `MeterName`. The API rejects `MeterName` outright
  and lists the valid dimensions in the 400 body.
- Azure Monitor metric names have changed: **`InputTokens`**, **`OutputTokens`**,
  **`ProvisionedUtilization`** — older material says `ProcessedPromptTokens`, `GeneratedTokens` and
  `AzureOpenAIProvisionedManagedUtilizationV2`. An `OpenAI`-kind resource may still publish the old
  names and an `AIServices`-kind one the new, so this notebook asks the resource which it has.
  Missing or null telemetry is not proof of inactivity; keep real zeros, not fabricated ones.
  Revised collectors require tenant acceptance testing; offline checks do not prove live refresh.

**Copilot Studio pay-as-you-go appears in this data too**, as `Pay As You Go Copilot Credit` at
the rate observed in the original sample. Compare like-for-like periods, currencies and scopes;
differences can reflect billing latency, discounts or adjustments as well as rate assumptions.

Authentication is explicit: an Entra application credential is read from Key Vault and exchanged
for an ARM token. Fabric's documented `getToken` audiences do **not** include ARM; do not assume
`getToken("https://management.azure.com/")` or an automatic notebook managed identity works.
The notebook execution identity needs Key Vault secret-read and Lakehouse write access; the
application needs Azure cost, resource-discovery and metrics read permissions. See the
[Fabric setup guide](../README.md#2b-azure-ai-foundry-tables-optional) for identities and scheduling.

Each successful run replaces the last `DAYS` complete UTC days, including empty results.
All collection and validation completes before either table is written. Refresh the semantic
model only after the entire notebook succeeds; the two Delta writes are not one transaction.


## Configure

In [ ]:
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"
TENANT_ID = "00000000-0000-0000-0000-000000000000"
CLIENT_ID = "00000000-0000-0000-0000-000000000000"
KEY_VAULT_URL = "https://<your-vault>.vault.azure.net/"
CLIENT_SECRET_NAME = "consumption-central-azure-client-secret"

# Days of history. Cost Management keeps far more; 90 is enough to see a trend
# without making the first run slow.
DAYS = 90

# Resource-tag keys to try, in order, for department attribution. Tenants name
# this differently and none of them being present is fine - DepartmentTag is
# optional everywhere it is used.
TAG_KEYS = ["Department", "department", "CostCentre", "CostCenter",
            "Team", "BusinessUnit", "Owner"]

LAKEHOUSE_SPEND  = "azure_ai_spend"
LAKEHOUSE_TOKENS = "azure_ai_tokens"


In [ ]:
import requests
import math
import time
from copy import deepcopy
from datetime import date, datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import urlsplit
from uuid import UUID

ARM   = "https://management.azure.com"
_token = None
_token_until = 0


def arm_headers():
    global _token, _token_until
    if _token is None or time.monotonic() >= _token_until:
        secret = notebookutils.credentials.getSecret(KEY_VAULT_URL, CLIENT_SECRET_NAME)
        r = requests.post(
            f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
            data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
                  "client_secret": secret, "scope": f"{ARM}/.default"},
            timeout=90, allow_redirects=False)
        r.raise_for_status()
        payload = r.json()
        _token = payload["access_token"]
        _token_until = time.monotonic() + max(0, int(payload["expires_in"]) - 300)
    return {"Authorization": f"Bearer {_token}", "Content-Type": "application/json"}


def retry_delay(headers, attempt):
    delays = [float(2 ** attempt)]
    for name, value in headers.items():
        key = name.lower()
        if key == "retry-after" or (key.startswith("x-ms-ratelimit-") and key.endswith("retry-after")):
            try:
                delay = float(value)
            except ValueError:
                if key != "retry-after":
                    raise ValueError("Invalid Cost Management retry header") from None
                retry_at = parsedate_to_datetime(value)
                if retry_at.tzinfo is None:
                    retry_at = retry_at.replace(tzinfo=timezone.utc)
                delay = (retry_at - datetime.now(timezone.utc)).total_seconds()
            if not math.isfinite(delay):
                raise ValueError("Invalid retry delay")
            delays.append(max(0, delay))
    delay = max(delays)
    if delay > 300:
        raise RuntimeError("Server retry delay exceeds 300 seconds; defer the job")
    return delay


def arm_request(method, url, **kwargs):
    # Continuation URLs are opaque, but must never receive credentials off ARM.
    parsed = urlsplit(url)
    if (parsed.scheme != "https" or parsed.netloc.lower() != "management.azure.com"
            or parsed.fragment):
        raise ValueError("Invalid ARM request or continuation URL")
    for attempt in range(5):
        r = requests.request(method, url, headers=arm_headers(), timeout=180,
                             allow_redirects=False, **kwargs)
        if r.status_code in (429, 502, 503, 504) and attempt < 4:
            delay = retry_delay(r.headers, attempt)
            print(f"ARM HTTP {r.status_code}; retrying the same request in {delay:g}s")
            r.close()
            time.sleep(delay)
            continue
        r.raise_for_status()
        if r.status_code not in (200, 204):
            raise RuntimeError(f"Unexpected ARM status {r.status_code}")
        return r
    raise RuntimeError("ARM retry budget exhausted")


def arm_get(url, **params):
    return arm_request("GET", url, params=params).json()


def arm_list(url, **params):
    items, seen = [], set()
    while url:
        if url in seen:
            raise RuntimeError("Repeated ARM continuation URL")
        seen.add(url)
        page = arm_get(url, **params)
        items.extend(page["value"])
        url, params = page.get("nextLink"), {}
    return items


def cost_query(url, body):
    records, seen = [], set()
    while url:
        if url in seen:
            raise RuntimeError("Repeated Cost Management continuation URL")
        seen.add(url)
        response = arm_request("POST", url, json=body)
        if response.status_code == 204:
            if records or len(seen) > 1:
                raise RuntimeError("Unexpected empty Cost Management continuation page")
            return []
        page = response.json()["properties"]
        columns = [c["name"] for c in page["columns"]]
        if len(columns) != len(set(columns)):
            raise ValueError("Duplicate cost columns")
        for values in page["rows"]:
            if len(values) != len(columns):
                raise ValueError("Cost row does not match its page columns")
            row = dict(zip(columns, values))
            for alias, spec in body["dataset"]["aggregation"].items():
                candidates = [spec["name"], alias]
                if spec["name"] == "Cost":
                    candidates.append("PreTaxCost")
                found = [key for key in candidates if key in row]
                if not found:
                    raise ValueError(f"Missing cost aggregation {spec['name']}")
                row[spec["name"]] = finite_number(row[found[0]])
            required = [g["name"] for g in body["dataset"]["grouping"]]
            for key in required + ["UsageDate", "Currency"]:
                if key not in row:
                    raise ValueError(f"Missing cost column {key}")
            records.append(row)
        url = page.get("nextLink")
    return records


def finite_number(value):
    number = float(value)
    if not math.isfinite(number):
        raise ValueError("Non-finite numeric value in Azure response")
    return number


def resource_group(resource_id):
    parts = resource_id.split("/")
    lower = [p.lower() for p in parts]
    return parts[lower.index("resourcegroups") + 1] if "resourcegroups" in lower else ""


## 1. Spend, from Cost Management

In [ ]:
for identifier in (SUBSCRIPTION_ID, TENANT_ID, CLIENT_ID):
    if UUID(identifier).int == 0:
        raise ValueError("Configure subscription, tenant and application client IDs before running")
if not isinstance(DAYS, int) or isinstance(DAYS, bool) or not 1 <= DAYS <= 90:
    raise ValueError("DAYS must be an integer from 1 to 90 (Monitor retention)")

end   = datetime.now(timezone.utc).date()
start = end - timedelta(days=DAYS)

body = {
    "type": "ActualCost",
    "timeframe": "Custom",
    "timePeriod": {"from": f"{start}T00:00:00Z",
                   "to": f"{end - timedelta(days=1)}T23:59:59Z"},
    "dataset": {
        "granularity": "Daily",
        "aggregation": {
            "totalCost": {"name": "Cost", "function": "Sum"},
            "totalQty":  {"name": "UsageQuantity", "function": "Sum"},
        },
        # The documented limit is two grouping clauses. Discover service/category
        # pairs first, then query Meter x ResourceId within each pair.
        "grouping": [{"type": "Dimension", "name": d} for d in
                     ["ServiceName", "MeterCategory"]],
        "filter": {"dimensions": {
            "name": "ServiceName",
            "operator": "In",
            # 'Foundry Models' is the real one. The others are included so a
            # tenant on older SKUs still returns something.
            "values": ["Foundry Models", "Microsoft Copilot Studio",
                       "Cognitive Services", "Azure Machine Learning",
                       "Azure OpenAI"],
        }},
    },
}

cost_url = (f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.CostManagement"
            "/query?api-version=2025-03-01")
summary = cost_query(cost_url, body)
pairs = {(r["ServiceName"], r["MeterCategory"]) for r in summary}
rows = []
for service, category in sorted(pairs):
    if not service or not category:
        raise ValueError("Cost query returned a blank service/category; cannot safely partition")
    detail = deepcopy(body)
    detail["dataset"]["grouping"] = [
        {"type": "Dimension", "name": d} for d in ["Meter", "ResourceId"]]
    detail["dataset"]["filter"] = {"and": [
        {"dimensions": {"name": key, "operator": "In", "values": [value]}}
        for key, value in [("ServiceName", service), ("MeterCategory", category)]]}
    details = cost_query(cost_url, detail)
    if not details:
        raise RuntimeError("Cost detail is empty for a discovered service/category; rerun")
    for row in details:
        row.update(ServiceName=service, MeterCategory=category)
    rows.extend(details)
print(f"{len(rows):,} cost rows")


In [ ]:
# Resource tags, fetched separately and joined on resource id. Grouping the
# cost query on a TagKey would be tidier but changes a query that is verified
# working.
tags = {}
res = arm_list(f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/resources",
              **{"api-version": "2021-04-01"})
for r in res:
    t = r.get("tags") or {}
    for k in TAG_KEYS:
        if k in t:
            tags[r["id"].lower()] = t[k]
            break
print(f"{len(tags):,} resources carry a department tag")


In [ ]:
recs = []
cost_keys = set()
for r in rows:
    rid   = str(r["ResourceId"] or "")
    parts = rid.split("/")
    usage_date = datetime.strptime(str(r["UsageDate"]), "%Y%m%d").date()
    if not start <= usage_date < end:
        raise ValueError("Cost date outside the requested UTC window")
    key = (usage_date, r["ServiceName"], r["MeterCategory"], r["Meter"],
           rid.lower(), r["Currency"])
    if key in cost_keys:
        raise ValueError("Duplicate aggregated cost grain; refusing to double count")
    cost_keys.add(key)
    recs.append({
        "UsageDate":     usage_date,
        "ServiceName":   r["ServiceName"],
        "MeterCategory": r["MeterCategory"],
        "Meter":         r["Meter"],
        "ResourceId":    rid,
        "ResourceName":  parts[-1] if parts else "",
        "ResourceGroup": resource_group(rid),
        "Cost":          r["Cost"],
        "UsageQuantity": r["UsageQuantity"],
        "Currency":      r["Currency"],
        "DepartmentTag": tags.get(rid.lower(), ""),
    })

print(f"validated {len(recs):,} spend rows; writes wait for metrics collection")


## 2. Tokens and provisioned utilisation, from Azure Monitor

Run this section even without provisioned deployments: token/request metrics can still exist.
Counters use `Total`; utilisation uses server-side daily `Average`. Only deployment is split,
with other dimensions rolled up by Azure, so averages are never summed or averaged again.
One name per equivalent metric family is selected; `TotalTokens` is a fallback only when
neither input nor output metrics exists. Do not sum all `Metric` values together.
Zero points are retained; absent/null points mean missing telemetry, not proven zero traffic.

In [ ]:
METRIC_FAMILIES = [
    ("InputTokens", "ProcessedPromptTokens"),
    ("OutputTokens", "GeneratedTokens"),
    ("ModelRequests", "AzureOpenAIRequests", "TotalCalls"),
    ("ProvisionedUtilization", "AzureOpenAIProvisionedManagedUtilizationV2"),
]
UTILIZATION = set(METRIC_FAMILIES[-1])
SERIES_LIMIT = 10000


def select_metrics(definitions, kind):
    available = {d["name"]["value"]: d for d in definitions}
    selected = []
    for family in METRIC_FAMILIES:
        name = next((n for n in family if n in available
                     and not (n == "TotalCalls" and kind.lower() == "openai")), None)
        if name:
            selected.append(name)
    if not any(n in selected for family in METRIC_FAMILIES[:2] for n in family):
        if "TotalTokens" in available:
            selected.append("TotalTokens")
    return [(name, available[name]) for name in selected]


def metric_parameters(name, definition, first, last):
    aggregation = "Average" if name in UTILIZATION else "Total"
    supported = definition.get("supportedAggregationTypes") or [
        definition["primaryAggregationType"]]
    if aggregation.lower() not in {s.lower() for s in supported}:
        raise ValueError(f"{name} does not support required {aggregation} aggregation")
    dimensions = [d["value"] for d in definition.get("dimensions", [])]
    deployment = next((d for candidate in ("modeldeploymentname", "deploymentname", "deployment")
                       for d in dimensions if d.lower() == candidate), None)
    params = {"api-version": "2023-10-01", "metricnames": name,
              "aggregation": aggregation, "interval": "P1D",
              "timespan": f"{first}T00:00:00Z/{last}T00:00:00Z",
              "AutoAdjustTimegrain": "false", "ValidateDimensions": "true"}
    if definition.get("namespace"):
        params["metricnamespace"] = definition["namespace"]
    if deployment:
        params.update({"$filter": f"{deployment} eq '*'", "top": SERIES_LIMIT})
    return params, deployment, aggregation.lower()


def metric_points(payload, name, deployment_dimension, field, account, first, last):
    if payload.get("error"):
        raise RuntimeError(f"Azure Monitor error: {payload['error']}")
    metrics = payload["value"]
    if len(metrics) != 1 or metrics[0]["name"]["value"] != name:
        raise ValueError(f"Azure Monitor omitted or changed requested metric {name}")
    metric = metrics[0]
    if metric.get("errorCode") not in (None, "Success", "0"):
        raise RuntimeError(f"{name}: {metric['errorCode']}: {metric.get('errorMessage', '')}")
    if payload.get("interval") != "P1D":
        raise ValueError(f"{name}: expected daily points; refusing to reaggregate averages")
    series = metric["timeseries"]
    if len(series) >= SERIES_LIMIT:
        raise RuntimeError(f"{name}: series limit reached; results may be truncated")
    result, seen_series, seen_points = [], set(), set()
    for series_item in series:
        metadata = {m["name"]["value"].lower(): m["value"]
                    for m in series_item.get("metadatavalues", [])}
        expected = deployment_dimension.lower() if deployment_dimension else None
        if any(key != expected and value not in ("", None) for key, value in metadata.items()):
            raise ValueError(f"{name}: unexpected dimension split; refusing to double count")
        if expected and expected not in metadata:
            raise ValueError(f"{name}: missing deployment metadata")
        deployment = metadata.get(expected, "") or ""
        if deployment == "*" or deployment in seen_series:
            raise ValueError(f"{name}: duplicate or rolled-up deployment series")
        seen_series.add(deployment)
        for point in series_item["data"]:
            stamp = datetime.fromisoformat(point["timeStamp"].replace("Z", "+00:00"))
            if stamp.tzinfo is None:
                raise ValueError("Metric timestamp is missing its timezone")
            stamp = stamp.astimezone(timezone.utc)
            day = stamp.date()
            # Some APIs include the upper endpoint; adjacent windows must not overlap.
            if day == last and stamp.hour == stamp.minute == stamp.second == stamp.microsecond == 0:
                continue
            if not first <= day < last or (stamp.hour, stamp.minute, stamp.second, stamp.microsecond) != (0, 0, 0, 0):
                raise ValueError(f"{name}: unexpected daily timestamp")
            if field not in point and any(k in point for k in ("total", "average", "count", "minimum", "maximum")):
                raise ValueError(f"{name}: missing requested {field} aggregation")
            value = point.get(field)
            if value is None:
                continue
            key = (day, deployment)
            if key in seen_points:
                raise ValueError(f"{name}: duplicate daily metric point")
            seen_points.add(key)
            result.append({"Date": day, "ResourceName": account["name"],
                           "ResourceGroup": resource_group(account["id"]),
                           "Deployment": deployment, "Metric": name,
                           "Value": finite_number(value)})
    return result


accounts = arm_list(
    f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers"
    f"/Microsoft.CognitiveServices/accounts",
    **{"api-version": "2023-05-01"})

points = []
account_ids = set()
for a in accounts:
    if a["id"].lower() in account_ids:
        raise ValueError("Duplicate Cognitive Services account in paginated inventory")
    account_ids.add(a["id"].lower())
    defs = arm_list(f"{ARM}{a['id']}/providers/microsoft.insights/metricDefinitions",
                    **{"api-version": "2018-01-01"})
    ask = select_metrics(defs, a.get("kind", ""))
    if not ask:
        print(f"  {a['name']}: publishes none of the known token metrics")
        continue
    print(f"  {a['name']}: {', '.join(name for name, _ in ask)}")
    for name, definition in ask:
        first = start
        while first < end:
            last = min(first + timedelta(days=30), end)
            params, dimension, field = metric_parameters(name, definition, first, last)
            md = arm_get(f"{ARM}{a['id']}/providers/microsoft.insights/metrics", **params)
            points.extend(metric_points(md, name, dimension, field, a, first, last))
            first = last

SPEND_SCHEMA = ("UsageDate date, ServiceName string, MeterCategory string, Meter string, "
                "ResourceId string, ResourceName string, ResourceGroup string, Cost double, "
                "UsageQuantity double, Currency string, DepartmentTag string")
TOKENS_SCHEMA = ("Date date, ResourceName string, ResourceGroup string, Deployment string, "
                 "Metric string, Value double")
spend_frame = spark.createDataFrame(recs, schema=SPEND_SCHEMA)
tokens_frame = spark.createDataFrame(points, schema=TOKENS_SCHEMA)
for frame, table, count in [(spend_frame, LAKEHOUSE_SPEND, len(recs)),
                            (tokens_frame, LAKEHOUSE_TOKENS, len(points))]:
    frame.write.mode("overwrite").format("delta").saveAsTable(table)
    print(f"wrote {table}: {count:,} rows")
if not points:
    print("No metric points returned. Missing telemetry is not proof of zero traffic.")
